# IRIS: 03 Metro Manila flood drill-down

Two barangays in the same city get the same rain but not the same flood. This notebook overlays the UP NOAH
100-year flood hazard map for Metro Manila on the 2025 barangay boundaries and the 2020 census ages, and estimates
how many people, and how many children under 5 and adults 65+, live in medium or high flood hazard zones.

- Flood hazard: UP NOAH Center, Metro Manila 100-year flood hazard (`Var` 1 = low, 2 = medium, 3 = high).
  Save the downloaded zip's contents as `datasets/MetroManila_Flood_100year.shp` (with its .dbf, .shx, .prj).
- People in a hazard zone are estimated by area share (population spread evenly over each barangay), which is
  a simplification: settlements are not spread evenly.

Output: `data/processed/ncr_barangays.geojson`. The overlay takes about 5 to 10 minutes.

In [1]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
warnings.filterwarnings('ignore')

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
RAW, OUT = ROOT / 'datasets', ROOT / 'data' / 'processed'
UTM = 32651   # metres, for areas

bgy = gpd.read_file(RAW / 'phl_admin_boundaries.gdb', layer='phl_admin4', where="adm1_pcode='PH13'")
bgy = bgy[['adm4_name', 'adm4_pcode', 'adm3_name', 'geometry']].to_crs(UTM)
bgy['area'] = bgy.area
flood = gpd.read_file(RAW / 'MetroManila_Flood_100year.shp').to_crs(UTM)
print(len(bgy), 'barangays;', flood['Var'].tolist(), 'hazard levels')

1712 barangays; [1.0, 2.0, 3.0] hazard levels


In [2]:
ov = gpd.overlay(bgy[['adm4_pcode', 'geometry']], flood, how='intersection', keep_geom_type=True)
ov['a'] = ov.area
share = ov.pivot_table(index='adm4_pcode', columns='Var', values='a', aggfunc='sum').fillna(0)
share.columns = [f'share_var{int(v)}' for v in share.columns]
bgy = bgy.merge(share, left_on='adm4_pcode', right_index=True, how='left')
for v in (1, 2, 3):
    bgy[f'share_var{v}'] = (bgy[f'share_var{v}'].fillna(0) / bgy['area']).clip(0, 1)
bgy[['share_var1', 'share_var2', 'share_var3']].describe().round(3)

,share_var1,share_var2,share_var3
count,1712.000,1712.000,1712.000
mean,0.209,0.359,0.058
std,0.165,0.307,0.129
min,0.000,0.000,0.000
25%,0.077,0.077,0.000
50%,0.172,0.275,0.000
75%,0.311,0.621,0.060
max,0.859,1.000,1.000


Census barangays are matched on code (`PH` + region + `0` + the rest of the old 9-digit code), and the City of
Manila's barangays, whose codes changed, by city and barangay name.

In [3]:
census = pd.read_excel(RAW / '2020-census-total-popn_single-age_malefemale_admin4.xlsx', engine='openpyxl')
census = census[census['BgyCode_old'].str.startswith('PH13')].copy()
U5 = ['Under 1_MF', '1_MF', '2_MF', '3_MF', '4_MF']
O65 = [f'{a}_MF' for a in range(65, 80)] + ['80 and Over_MF']
census[U5 + O65 + ['Total_MF']] = census[U5 + O65 + ['Total_MF']].replace('-', 0).apply(pd.to_numeric).fillna(0)
census['under5'], census['over65'] = census[U5].sum(axis=1), census[O65].sum(axis=1)
MANILA = {'TONDO I/II', 'BINONDO', 'QUIAPO', 'SAN NICOLAS', 'SANTA CRUZ', 'SAMPALOC', 'SAN MIGUEL', 'ERMITA',
          'INTRAMUROS', 'MALATE', 'PACO', 'PANDACAN', 'PORT AREA', 'SANTA ANA'}
norm = lambda s: re.sub(r'[^a-z0-9]', '', str(s).lower().replace('barangay', '').replace('poblacion', 'pob'))
city = lambda s: s.str.upper().str.replace('CITY OF ', '').str.replace(' CITY', '')
census['city'] = np.where(census['Mun'].isin(MANILA), 'MANILA', city(census['Mun']))
census['nk'] = census['city'] + '|' + census['Bgy'].map(norm)
census['key'] = 'PH' + census['BgyCode_old'].str[2:4] + '0' + census['BgyCode_old'].str[4:]
bgy['nk'] = city(bgy['adm3_name']) + '|' + bgy['adm4_name'].map(norm)

cols = ['Total_MF', 'under5', 'over65']
bgy = bgy.merge(census[['key'] + cols], left_on='adm4_pcode', right_on='key', how='left')
by_name = census.drop_duplicates('nk').set_index('nk')[cols]
miss = bgy['Total_MF'].isna()
for c in cols:
    bgy.loc[miss, c] = bgy.loc[miss, 'nk'].map(by_name[c])
print('Unmatched:', bgy.loc[bgy['Total_MF'].isna(), 'adm4_name'].tolist())   # cemetery and mall areas, no residents

Unmatched: ['Manila North Cemetery', 'Tutuban Mall (Claimed by Five Barangays of Tondo, Manila)']


In [4]:
b = bgy[bgy['Total_MF'].notna()].copy()
b['vulnerable'] = b['under5'] + b['over65']
b['flood_share_any'] = (b['share_var1'] + b['share_var2'] + b['share_var3']).clip(0, 1)
b['flood_share_med_high'] = (b['share_var2'] + b['share_var3']).clip(0, 1)
b['people_med_high'] = (b['Total_MF'] * b['flood_share_med_high']).round(0)
b['vulnerable_med_high'] = (b['vulnerable'] * b['flood_share_med_high']).round(0)
b['rank'] = b['vulnerable_med_high'].rank(ascending=False, method='min').astype(int)
b['geometry'] = b.simplify(12, preserve_topology=True)
out = b.rename(columns={'adm4_pcode': 'bgy_code', 'adm4_name': 'barangay', 'adm3_name': 'city',
                        'Total_MF': 'population'})[
    ['bgy_code', 'barangay', 'city', 'population', 'under5', 'over65', 'vulnerable', 'share_var1', 'share_var2',
     'share_var3', 'flood_share_any', 'flood_share_med_high', 'people_med_high', 'vulnerable_med_high', 'rank',
     'geometry']].to_crs(4326)
for c in ['share_var1', 'share_var2', 'share_var3', 'flood_share_any', 'flood_share_med_high']:
    out[c] = out[c].round(3)
for c in ['population', 'under5', 'over65', 'vulnerable']:
    out[c] = out[c].astype(int)
out.to_file(OUT / 'ncr_barangays.geojson', driver='GeoJSON', COORDINATE_PRECISION=5)
print(f"{out['people_med_high'].sum():,.0f} people, {out['vulnerable_med_high'].sum():,.0f} of them under 5 or 65+, "
      "live in medium or high flood hazard zones")
out.sort_values('rank').head(10)[['barangay', 'city', 'population', 'flood_share_med_high', 'vulnerable_med_high']]

4,244,429 people, 606,567 of them under 5 or 65+, live in medium or high flood hazard zones


,barangay,city,population,flood_share_med_high,vulnerable_med_high
956,Pinagbuhatan,City of Pasig,163598,0.475,11433.0
952,Manggahan,City of Pasig,88078,0.731,9528.0
958,Rosario,City of Pasig,73979,0.870,9103.0
930,Malanday,City of Marikina,53886,0.930,7520.0
1110,Batasan Hills,Quezon City,166572,0.310,7444.0
1460,San Dionisio,City of Parañaque,72522,0.609,6626.0
968,Santolan,City of Pasig,57933,0.757,6459.0
931,Nangka,City of Marikina,43368,0.956,6360.0
953,Maybunga,City of Pasig,45555,0.935,6191.0
981,Bagong Silangan,Quezon City,106886,0.367,5983.0
